# Pipeline A — which vision model is best?

Same question for every model (the "evidence" prompt: first name each line in the field, then score 1-5),
on the **same 150 Missouri collages** (75 gully + 75 no gully). Every model is reported, not just the winner.

| Model | Size | Why it's here |
|---|---|---|
| `qwen2.5vl:7b` | 7B | what we used so far (baseline) |
| `qwen3-vl:8b` | 8B | newer version of the same family, same size |
| `gemma4:26b` | 26B (fast, mixture-of-experts) | strong and quick |
| `gemma4:31b` | 31B | strongest model that fits Kaggle's free GPUs |

**Two modes** (cell 5):
- `MODE = "test"`: all models on 150 Missouri collages. Roughly 3-5 h in total.
- `MODE = "full"`: only `BEST_MODEL`, on all 621 Missouri + 300 Hisar.

**If a run gets cut off:** right panel → **Add Input** → **Your Work** → pick this notebook (its last version)
→ **Save & Run All** again. Cell 4 copies the finished CSVs back in and only the missing part is re-run.

**How to run so files are KEPT:** right panel → GPU T4 x2, Internet On, Input `gully-collages` →
**Save Version → Save & Run All (Commit)**. Close the browser if you like. Later, open the notebook page
(the address without `/edit`) → **Output** tab → download the CSVs.

In [ ]:
# 1. GPU check
!nvidia-smi

In [ ]:
# 2. Install Ollama
!apt-get -qq update > /dev/null && apt-get -qq install -y zstd pciutils lshw > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh

In [ ]:
# 3. Start Ollama server
import subprocess, time, requests
subprocess.Popen("ollama serve > /kaggle/working/ollama.log 2>&1", shell=True)
for _ in range(60):
    try:
        requests.get("http://localhost:11434", timeout=2); print("Ollama server is running"); break
    except Exception:
        time.sleep(2)
else:
    raise RuntimeError("Ollama did not start - see /kaggle/working/ollama.log")
!ollama --version

In [ ]:
# 4. Settings + data
import base64, csv, glob, json, os
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

MODE = "test"                 # "test" = compare models on 150 Missouri; "full" = BEST_MODEL on everything
MODELS = ["llava-llama3:8b", "llama3.2-vision:11b", "qwen2.5vl:7b", "qwen3-vl:8b", "gemma4:26b", "gemma4:31b"]
BEST_MODEL = "gemma4:31b"     # set this after the test, to the model with the highest AUC
TEST_PER_CLASS = 75

OLLAMA = "http://localhost:11434"
OUT_DIR = "/kaggle/working"

# ── CHECKPOINT RESTORE ──
# If you added a previous run of this notebook as Input, copy its finished CSVs here,
# so models / images that are already done are skipped instead of re-run.
import shutil
for src in glob.glob("/kaggle/input/**/pipeline_a_*.csv", recursive=True):
    dst = os.path.join(OUT_DIR, os.path.basename(src))
    n_src = len(pd.read_csv(src))
    n_dst = len(pd.read_csv(dst)) if os.path.exists(dst) else -1
    if n_src > n_dst:
        shutil.copy(src, dst)
        print(f"Restored checkpoint: {os.path.basename(src)} ({n_src} rows)")

def find_dataset(name):
    hits = [p for p in glob.glob("/kaggle/input/**/collages.csv", recursive=True) if name in p.lower()]
    if not hits:
        raise FileNotFoundError(f"No collages.csv for '{name}' - did you add the gully-collages dataset?")
    folder = os.path.dirname(sorted(hits, key=len)[0])
    df = pd.read_csv(folder + "/collages.csv", dtype={"label": str})
    df["path"] = folder + "/" + df["collage"]
    df = df[df["path"].map(os.path.exists)].reset_index(drop=True)
    print(f"{name}: {len(df)} collages")
    return df

missouri = find_dataset("missouri")
hisar = find_dataset("hisar")      # collages_hisar (rebuilt T43REN dataset)
if MODE == "test":
    missouri = (missouri[missouri["label"].isin(["0", "1"])]
                .groupby("label").sample(n=TEST_PER_CLASS, random_state=42)   # same 150 as before
                .reset_index(drop=True))
    print("Test subset:", missouri["label"].value_counts().to_dict())

In [ ]:
# 5. The question (the 'evidence' prompt that worked best) + helpers
PROMPT = ('The image is a collage of {n} overhead images of the SAME agricultural field, taken on different dates. '
          'Each tile has its date or image number in the top-left corner. Ignore any grey "no image" tile or text tile.\n\n'
          'An ephemeral gully is a small erosion channel cut by rainwater runoff, which farmers fill in during tillage. '
          'Many fields have no gully, so only give a high score if you see clear evidence.\n\n'
          'First, list the linear features you can see (if any) and say what each most likely is, '
          'e.g. crop rows, field boundary, road, tractor track, canal, or erosion channel. Then give your score.\n\n'
          'Score from 1 to 5 how likely it is that this field contains an ephemeral gully:\n'
          '1 = almost certainly no, 2 = probably no, 3 = unsure, 4 = probably yes, 5 = almost certainly yes.')
SCHEMA = {"type": "object",
          "properties": {"features": {"type": "string"},
                         "score": {"type": "integer", "enum": [1, 2, 3, 4, 5]}},
          "required": ["features", "score"]}

def pull(model):
    print(f"Downloading {model} ...", flush=True)
    r = requests.post(f"{OLLAMA}/api/pull", json={"model": model, "stream": False}, timeout=3600)
    r.raise_for_status()
    print("  done:", r.json().get("status"))

def remove(model):
    # free disk space before the next model
    requests.delete(f"{OLLAMA}/api/delete", json={"model": model}, timeout=120)

def ask(model, image_path, n_images):
    with open(image_path, "rb") as f:
        img = base64.b64encode(f.read()).decode()
    body = {"model": model,
            "messages": [{"role": "user", "content": PROMPT.format(n=n_images), "images": [img]}],
            "format": SCHEMA, "stream": False, "think": False,
            "options": {"temperature": 0, "num_ctx": 8192, "seed": 42}}
    r = requests.post(f"{OLLAMA}/api/chat", json=body, timeout=900)
    if r.status_code == 400 and "think" in r.text.lower():      # model has no thinking switch
        body.pop("think"); r = requests.post(f"{OLLAMA}/api/chat", json=body, timeout=900)
    r.raise_for_status()
    raw = r.json()["message"]["content"]
    try:
        out = json.loads(raw)
        return int(out["score"]), str(out.get("features", "")), raw
    except Exception:
        return "", raw, raw

def csv_path(model, name):
    return f"{OUT_DIR}/pipeline_a_{model.replace(':', '-').replace('/', '-')}_{name}.csv"

def is_done(model, name, df):
    p = csv_path(model, name)
    return os.path.exists(p) and set(df["tile_id"]) <= set(pd.read_csv(p)["tile_id"])

def run(model, name, df):
    out_csv = csv_path(model, name)
    done = set(pd.read_csv(out_csv)["tile_id"]) if os.path.exists(out_csv) else set()
    todo = df[~df["tile_id"].isin(done)]
    new_file = not os.path.exists(out_csv)
    with open(out_csv, "a", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        if new_file:
            w.writerow(["tile_id", "label", "score", "reason", "seconds", "raw"])
        for _, row in tqdm(todo.iterrows(), total=len(todo), desc=f"{model} / {name}"):
            t = time.time()
            try:
                s, reason, raw = ask(model, row["path"], row["n_images"])
            except Exception as e:
                s, reason, raw = "", f"ERROR: {e}", ""
            w.writerow([row["tile_id"], row["label"] if pd.notna(row["label"]) else "", s, reason,
                        round(time.time() - t, 1), raw])
            f.flush()
    return pd.read_csv(out_csv, dtype={"label": str})

In [ ]:
# 6. Run
from sklearn.metrics import roc_auc_score, balanced_accuracy_score, accuracy_score, f1_score

def evaluate(df):
    d = df[df["label"].isin(["0", "1"]) & pd.to_numeric(df["score"], errors="coerce").notna()]
    y, s = d["label"].astype(int).values, d["score"].astype(int).values
    if len(d) == 0 or len(set(y)) < 2:
        return None
    best = max(((t, balanced_accuracy_score(y, (s >= t).astype(int)), accuracy_score(y, (s >= t).astype(int)),
                 f1_score(y, (s >= t).astype(int), zero_division=0), (s >= t).mean()) for t in [2, 3, 4, 5]),
               key=lambda r: r[1])
    return {"scored": f"{len(d)}/{len(df)}", "AUC": round(roc_auc_score(y, s), 3), "threshold": best[0],
            "balanced_acc": round(best[1], 3), "acc": round(best[2], 3), "F1": round(best[3], 3),
            "yes_rate": round(best[4], 2), "sec_per_img": round(df["seconds"].median(), 1),
            "score_counts": pd.Series(s).value_counts().sort_index().to_dict()}

rows = []
models_to_run = MODELS if MODE == "test" else [BEST_MODEL]
for model in models_to_run:
    print(f"\n========== {model} ==========")
    try:
        if is_done(model, "missouri", missouri) and (MODE == "test" or is_done(model, "hisar", hisar)):
            print("  already finished (checkpoint) - skipping download")
        else:
            pull(model)
        res_m = run(model, "missouri", missouri)
        rows.append({"model": model, **(evaluate(res_m) or {"scored": "failed"})})
        if MODE == "full":
            run(model, "hisar", hisar)
        if MODE == "test":
            remove(model)
    except Exception as e:
        print("  FAILED:", e)
        rows.append({"model": model, "scored": f"failed: {e}"[:80]})
    pd.DataFrame(rows).to_csv(f"{OUT_DIR}/model_comparison_{MODE}.csv", index=False)

In [ ]:
# 7. Results
table = pd.DataFrame(rows)
print("=== Missouri: which model can tell gully from no-gully? ===")
print("AUC: 0.5 = guessing, 1.0 = perfect.  balanced_acc: 0.5 = guessing.\n")
print(table.to_string(index=False))
print("\nEarlier result, qwen2.5vl:7b with yes/no question: balanced acc 0.521, yes-rate 0.96")
print("Paper (Qwen2-VL-72B, yes/no): ~0.71 accuracy")

if MODE == "full":
    thr = rows[0].get("threshold", 4)
    tag = BEST_MODEL.replace(":", "-").replace("/", "-")
    h = pd.read_csv(f"{OUT_DIR}/pipeline_a_{tag}_hisar.csv")
    hs = pd.to_numeric(h["score"], errors="coerce")
    print(f"\n=== Hisar with {BEST_MODEL} (threshold {thr}, chosen on Missouri) ===")
    print("score counts:", hs.value_counts().sort_index().to_dict())
    print(f"predicted gully: {(hs >= thr).sum()} / {hs.notna().sum()}")